# Gen AI Evaluation Service

Agent Platform の Gen AI 評価サービス（Gen AI evaluation service）を使用すると、特定の要件に従ってあらゆる生成 AI モデルやアプリケーションを評価できます。

一般的なリーダーボードやレポートのみに頼るのではなく、独自の評価基準を定義し、独自のニーズ、ユースケース、データに対してさまざまなモデルがどのように機能するかを直接比較できます。

このサービスにより、以下のことが可能になります。

- モデルのパフォーマンスをより深く理解する。一般的な指標を超えて、モデルが特定のデータやタスクをどのように処理するかを把握します。
- 開発ライフサイクル全体を通じて情報に基づいた意思決定を行う。評価を活用してモデルの選択を導き、プロンプト エンジニアリングを改善し、モデルのカスタマイズを最適化します。
- Agent Platform の統合ツールを活用して評価を簡単に起動し、必要に応じて再利用することで、評価ワークフローを効率化する。

基本的に、生成 AI の評価を自ら制御できるようになり、選択したモデルが特定のアプリケーションに最適であることを確認できます。

## 学習目標

このノートブックでは、以下の内容を学びます。

- Gen AI Evaluation Service の使用方法
- さまざまな種類の評価手法（計算ベースおよびモデルベース）
- データセットを準備して評価に使用できるようにする方法
- 評価結果を分析して理解する

### セットアップ

In [ ]:
# 一般的なインポート
import os
import random
import string

import pandas as pd
from IPython.display import Markdown, display

# メイン
from vertexai.evaluation import (
    EvalTask,
    MetricPromptTemplateExamples,
    PairwiseMetric,
    PointwiseMetric,
    PointwiseMetricPromptTemplate,
)

from google import genai

### ヘルパー関数

この関数は、評価 SDK からの結果を読みやすい形式で表示するのに役立ちます。

In [ ]:
def display_eval_report(eval_result, metrics=None):
    """評価結果を表示します。"""

    title, summary_metrics, report_df = eval_result
    metrics_df = pd.DataFrame.from_dict(summary_metrics, orient="index").T
    if metrics:
        metrics_df = metrics_df.filter(
            [
                metric
                for metric in metrics_df.columns
                if any(selected_metric in metric for selected_metric in metrics)
            ]
        )
        report_df = report_df.filter(
            [
                metric
                for metric in report_df.columns
                if any(selected_metric in metric for selected_metric in metrics)
            ]
        )

    # 強調のために Markdown でタイトルを表示
    display(Markdown(f"## {title}"))

    # 指標の DataFrame を表示
    display(Markdown("### Summary Metrics"))
    display(metrics_df)

    # 詳細レポートの DataFrame を表示
    display(Markdown("### Report Metrics"))
    display(report_df)

## 評価プロセス

Agent Platform の Gen AI 評価サービスにより、特定のニーズと基準に基づいて任意の生成モデルを評価できます。

評価を支援するために従う 4 つのステップは以下のとおりです。

1. 定義（Define）: ビジネス目標に合わせて指標を調整し、評価アプローチ（計算ベース vs モデルベース）を選択します。
2. 準備（Prepare）: 実際のユースケースを反映したデータセットを作成します。
3. 実行（Run）: テンプレートや既存のサンプルを使用して簡単に評価を開始します。モデルを定義し、Agent Platform 内で再利用可能な `EvalTask` を作成して他の評価でも使用できるようにします。
4. 分析（Analyze）: 結果を解釈し、各モデルが特定の基準に対してどのように機能するかを理解します。

これらのステップを 1 つずつ見ていきましょう。

### 計算ベースの指標（Computation Based Metrics）

[計算ベースの指標](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/determine-eval#computation-based-metrics) は、数学的な公式を使用してモデルの出力を**グラウンドトゥルース（正解）または参照（reference）**と比較することによって計算されます。一般的に使用される計算ベースの指標には ROUGE や BLEU があります。一般的に使用される指標は、以下のグループに分類できます。

- 語彙ベースの指標（Lexicon-based metrics）: `Exact Match` や `ROUGE` など、LLM によって生成された結果と正解との間の文字列の類似度を数学的に計算します。
- カウントベースの指標（Count-based metrics）: `F1-score`、`Accuracy`、`Tool Name Match` など、特定の正解ラベルと一致した（ヒット）または一致しなかった（ミス）行数を集計します。
- 埋め込みベースの指標（Embedding-based metrics）: 埋め込み空間における LLM 生成結果と正解との間の距離を計算し、類似度レベルを反映します。これらには `cosine similarity` や `euclidean distance` が含まれます。

ここでは、Gemini を使用した要約において、異なるプロンプトがどれだけ適切に機能するかを評価することを考えます。まず `context` にいくつかの記事を定義し、評価に計算ベースの指標を使用するため、要約の正解データも定義する必要があります。これは `reference` で定義されます。`eval_dataset` は、評価に必要な列を含む DataFrame である必要があります。

In [ ]:
instruction = "Summarize the following article"

context = [
    "To make a classic spaghetti carbonara, start by bringing a large pot of salted water to a boil. While the water is heating up, cook pancetta or guanciale in a skillet with olive oil over medium heat until it's crispy and golden brown. Once the pancetta is done, remove it from the skillet and set it aside. In the same skillet, whisk together eggs, grated Parmesan cheese, and black pepper to make the sauce. When the pasta is cooked al dente, drain it and immediately toss it in the skillet with the egg mixture, adding a splash of the pasta cooking water to create a creamy sauce.",
    "Preparing a perfect risotto requires patience and attention to detail. Begin by heating butter in a large, heavy-bottomed pot over medium heat. Add finely chopped onions and minced garlic to the pot, and cook until they're soft and translucent, about 5 minutes. Next, add Arborio rice to the pot and cook, stirring constantly, until the grains are coated with the butter and begin to toast slightly. Pour in a splash of white wine and cook until it's absorbed. From there, gradually add hot chicken or vegetable broth to the rice, stirring frequently, until the risotto is creamy and the rice is tender with a slight bite.",
    "To bake a decadent chocolate cake from scratch, start by preheating your oven to 350°F (175°C) and greasing and flouring two 9-inch round cake pans. In a large mixing bowl, cream together softened butter and granulated sugar until light and fluffy. Beat in eggs one at a time, making sure each egg is fully incorporated before adding the next. In a separate bowl, sift together all-purpose flour, cocoa powder, baking powder, baking soda, and salt. Divide the batter evenly between the prepared cake pans and bake for 25-30 minutes, or until a toothpick inserted into the center comes out clean.",
]

reference = [
    "The process of making spaghetti carbonara involves boiling pasta, crisping pancetta or guanciale, whisking together eggs and Parmesan cheese, and tossing everything together to create a creamy sauce.",
    "Preparing risotto entails sautéing onions and garlic, toasting Arborio rice, adding wine and broth gradually, and stirring until creamy and tender.",
    "Baking a decadent chocolate cake requires creaming butter and sugar, beating in eggs and alternating dry ingredients with buttermilk before baking until done.",
]

eval_dataset = pd.DataFrame(
    {
        "context": context,
        "reference": reference,
        "instruction": [instruction] * len(context),
    }
)

次に、評価するさまざまなプロンプト テンプレートを定義します。このリストには、実験したいすべてのプロンプト テンプレートを含めることができます。デモンストレーションのために、ここでは以下の 2 つの異なるプロンプト テンプレートを使用します。

In [ ]:
prompt_templates = [
    "Instruction: {instruction}. Article: {context}. Summary:",
    "Article: {context}. Complete this task: {instruction}, in one sentence. Summary:",
]

次に、このタスクで測定するさまざまな指標を定義します。タスクに基づいて Agent Platform によって定義されているさまざまな指標があります。各種指標を確認するには [ドキュメント](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/determine-eval#computation-based-metrics) を参照してください。また、それぞれの正しい文字列値については [API ドキュメント](https://docs.cloud.google.com/python/docs/reference/vertexai/latest/vertexai.evaluation.EvalTask) を参照してください。

In [ ]:
metrics = [
    "rouge_1",
    "rouge_l_sum",
    "bleu",
    "safety",
]

モデル名（例: `'gemini-3.8-flash'`）を受け取り、指定されたモデルを呼び出して応答を返す関数を定義しましょう。

In [ ]:
# 標準の Google GenAI クライアントを初期化
client = genai.Client(vertexai=True, location="global")

def create_model_fn(model_name: str):
    """
    指定された model_name に合わせた model_function を生成して返します。
    """
    def model_function(prompt: str) -> str:
        # ターゲット モデルを使用してコンテンツを生成
        response = client.models.generate_content(
            model=model_name,
            contents=prompt,
        )

        # 生のテキスト文字列の応答を返す
        return response.text

    return model_function

model_func_3_8_flash = create_model_fn("gemini-3.8-flash")

すべてのパラメータを設定できたので、評価を開始する準備が整いました。[EvalTask](https://docs.cloud.google.com/python/docs/reference/vertexai/latest/vertexai.evaluation.EvalTask) を使用して、評価用の Agent Platform Experiment を開始します。

#### EvalTask クラスの理解

`EvalTask` クラスは、Gen AI Evaluation Service SDK フレームワークの中核コンポーネントです。Gen AI モデル / アプリケーションに対する評価ジョブを定義して実行でき、特定のタスクに対するパフォーマンスを測定するための構造化された方法を提供します。`EvalTask` は、評価プロセスの設計図と考えてください。評価タスクには、評価データセットと評価する指標のリストが含まれている必要があります。サポートされている指標については、Generative AI の [評価指標の定義ページ](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/determine-eval) に記載されています。データセットには、`pandas.DataFrame`、Python 辞書、またはファイル パスの URI を指定でき、`prompt`、`reference`、`response`、`baseline_model_response` などのデフォルトの列名を含めることができます。

- Bring-your-own-response（BYOR）: 評価したいデータがすでにデータセットに保存されています。`response_column_name` や `baseline_model_response_column_name` などのパラメータを使用するか、`metric_column_mapping` を通じて、独自のモデルとベースライン モデルの両方の応答列名をカスタマイズできます。

- プロンプト テンプレートなしでのモデル推論の実行: モデルへの入力プロンプトを含むデータセットがあり、評価前にモデル推論を実行したい場合です。評価データセットには `prompt` という名前の列が必要であり、モデルへの入力として直接使用されます。

- プロンプト テンプレートを使用したモデル推論の実行: プロンプト テンプレートへの入力変数を含むデータセットがあり、モデル推論用のプロンプトを組み立てたい場合です。評価データセットには、プロンプト テンプレート内の変数名に対応する列名が含まれている必要があります。たとえば、プロンプト テンプレートが "Instruction: {instruction}, context: {context}" の場合、データセットには `instruction` 列と `context` 列が含まれている必要があります。

`EvalTask` は、BYOR、Gemini モデルを使用したモデル推論、サードパーティ モデルのエンドポイント / SDK クライアント、またはカスタム モデル生成関数など、計算ベースの指標、モデルベースのポイントワイズ指標およびペアワイズ指標を使用した幅広い評価シナリオをサポートしています。`evaluate()` メソッドは評価プロセスをトリガーし、オプションでモデル、プロンプト テンプレート、実験ロギング設定、およびその他の評価実行構成を受け取ります。詳細については、[Gen AI Evaluation](https://docs.cloud.google.com/python/docs/reference/vertexai/latest/vertexai.evaluation) パッケージの SDK リファレンス ドキュメントを参照してください。

まずは、プロンプト テンプレートを使用したモデル推論から始めます。

In [ ]:
experiment_name = "prompt-engineering-eval"

summarization_eval_task = EvalTask(
    dataset=eval_dataset,
    metrics=metrics,
    experiment=experiment_name,
)

注意: 下のセルは実行完了までに 10 〜 15 分かかります。

In [ ]:
def generate_uuid(length: int = 8) -> str:
    """指定された長さ（デフォルトは 8）の UUID を生成します。"""
    return "".join(
        random.choices(string.ascii_lowercase + string.digits, k=length)
    )


run_id = generate_uuid()
eval_results = []


for i, prompt_template in enumerate(prompt_templates):
    experiment_run_name = f"eval-prompt-engineering-{run_id}-prompt-{i}"

    eval_result = summarization_eval_task.evaluate(
        prompt_template=prompt_template,
        experiment_run_name=experiment_run_name,
        model=model_func_3_8_flash,
    )

    eval_results.append(
        (f"Prompt #{i}", eval_result.summary_metrics, eval_result.metrics_table)
    )

In [ ]:
for eval_result in eval_results:
    display_eval_report(eval_result)

以下の関数を使用して、実験の実行結果を表示できます。

In [ ]:
summarization_eval_task.display_runs()

### モデルベースの指標（Model Based Metrics）

ユースケースに対して参照結果や正解データがない場合は、別の LLM を使用してさまざまなモデルを評価および比較できます。この手法は**モデルベース評価（Model based evaluation）**と呼ばれます。モデルベースの指標は、審査員（ジャッジ）モデルと照らし合わせて候補モデルを評価します。ほとんどのユースケースでジャッジ モデルは Gemini ですが、翻訳ユースケースの場合は [MetricX](https://github.com/google-research/metricx) や [COMET](https://huggingface.co/Unbabel/wmt22-comet-da) などのモデルを使用することもできます。

モデルベースの指標は、ポイントワイズ（Pointwise）またはペアワイズ（Pairwise）で測定できます。

- **ポイントワイズ指標（Pointwise metrics）**: ジャッジ モデルに評価基準に基づいて候補モデルの出力を個別に評価させます。たとえばスコアは 0 から 5 であり、0 は応答が基準に合致していないことを意味し、5 は応答が基準によく合致していることを意味します。

- **ペアワイズ指標（Pairwise metrics）**: ジャッジ モデルに 2 つのモデルの応答を比較させ、より優れた方を選択させます。これは、候補モデルとベースライン モデルを比較する際によく使用されます。ペアワイズ指標は、Gemini をジャッジ モデルとして使用する場合にのみサポートされます。

#### ポイントワイズ指標（Pointwise Metrics）

プロンプトを決定したところで、生成された要約を「fluency（流暢さ）」と「entertaining（面白さ）」という 2 つの定性的基準で評価したいとします。これら 2 つの基準を使用して、`text_quality` という指標を定義します。

In [ ]:
# text_quality の独自定義
metric_prompt_template = PointwiseMetricPromptTemplate(
    criteria={
        "fluency": "Sentences flow smoothly and are easy to read, avoiding awkward phrasing or run-on sentences. Ideas and sentences connect logically, using transitions effectively where needed.",
        "entertaining": "Short, amusing text that incorporates emojis, exclamations and questions to convey quick and spontaneous communication and diversion.",
    },
    rating_rubric={
        "2": "The response performs well on both criteria.",
        "1": "The response is somewhat aligned with both criteria",
        "0": "The response falls short on both criteria",
    },
)

text_quality = PointwiseMetric(
    metric="text_quality",
    metric_prompt_template=metric_prompt_template,
)

この `text_quality` 指標がどのようなものか確認してみましょう。

In [ ]:
print(text_quality.metric_prompt_template)

まず、対象のモデル（`gemini-3.8-flash`）から応答を取得し、リストに保存する必要があります。

In [ ]:
context = [
    "To make a classic spaghetti carbonara, start by bringing a large pot of salted water to a boil. While the water is heating up, cook pancetta or guanciale in a skillet with olive oil over medium heat until it's crispy and golden brown. Once the pancetta is done, remove it from the skillet and set it aside. In the same skillet, whisk together eggs, grated Parmesan cheese, and black pepper to make the sauce. When the pasta is cooked al dente, drain it and immediately toss it in the skillet with the egg mixture, adding a splash of the pasta cooking water to create a creamy sauce.",
    "Preparing a perfect risotto requires patience and attention to detail. Begin by heating butter in a large, heavy-bottomed pot over medium heat. Add finely chopped onions and minced garlic to the pot, and cook until they're soft and translucent, about 5 minutes. Next, add Arborio rice to the pot and cook, stirring constantly, until the grains are coated with the butter and begin to toast slightly. Pour in a splash of white wine and cook until it's absorbed. From there, gradually add hot chicken or vegetable broth to the rice, stirring frequently, until the risotto is creamy and the rice is tender with a slight bite.",
    "For a flavorful grilled steak, start by choosing a well-marbled cut of beef like ribeye or New York strip. Season the steak generously with kosher salt and freshly ground black pepper on both sides, pressing the seasoning into the meat. Preheat a grill to high heat and brush the grates with oil to prevent sticking. Place the seasoned steak on the grill and cook for about 4-5 minutes on each side for medium-rare, or adjust the cooking time to your desired level of doneness. Let the steak rest for a few minutes before slicing against the grain and serving.",
]

In [ ]:
responses = []
instruction = "Summarize the following article"
for article in context:
    prompt = f"Instruction: {instruction}. Article: {article}. Summary:"
    response = model_func_3_8_flash(prompt)
    responses.append(response)
responses

In [ ]:
eval_dataset = pd.DataFrame(
    {
        "response": responses,
    }
)

In [ ]:
EXPERIMENT_NAME = "pointwise-eval"
eval_task = EvalTask(
    dataset=eval_dataset, metrics=[text_quality], experiment=EXPERIMENT_NAME
)

pointwise_eval_results = eval_task.evaluate()

ここで 3 つの記事すべてに対する `summary_metrics` を確認できます。

In [ ]:
pointwise_eval_results.summary_metrics

上記で定義した基準に基づくと応答が両方の基準にある程度しか一致していないため、text_quality が `0.0` になっていることがわかります。`text_quality` で定義した基準に基づく各記事のスコアを取得するには、`metrics_table` にアクセスします。

In [ ]:
pointwise_eval_results.metrics_table

定義した基準に基づくと、モデルは各要約に対して中立的なスコアを付けたようです。モデルのスコアの根拠は、`text_quality/explanation` 列で確認できます。

#### ペアワイズ指標 - サイドバイサイド（SxS）でのモデル比較

プロンプトを決定したところで、2 つの異なる LLM（Gemini 3.8 Flash vs Gemini 3.1 Pro）から生成された要約も評価したいとします。ペアワイズ モデル評価とサイドバイサイド（並行）比較を使用して、2 つの異なるモデルからの要約を評価できます。

2 つのモデルを直接比較するには、`EvalTask` の実行内で `PairwiseMetric` を定義します。このアプローチにより、モデルのパフォーマンスを直接対決（head-to-head）で評価できます。

In [ ]:
instruction = "Summarize the following article"
prompt_template = "{instruction}. Article: {context}. Summary:"

pairwise_eval_dataset = pd.DataFrame(
    {
        "context": context,
        "instruction": [instruction] * len(context),
        "reference": reference,
    }
)

評価データセットの準備ができたら、比較する 2 つのモデルを定義します。`model_a` は `gemini-3.1-pro-preview`、`model_b` は `gemini-3.8-flash` とします。

In [ ]:
model_a = create_model_fn("gemini-3.1-pro-preview")
model_b = create_model_fn("gemini-3.8-flash")

上記のポイントワイズ評価で評価基準を定義したのと同様に、評価プロンプトをカスタマイズできます。ただし、時間を節約するために、Agent Platform は使用可能な [事前定義された評価プロンプト](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/metrics-templates) を `MetricPromptTemplateExamples` で提供しています。このユースケースでは、2 つのモデル間のテキスト品質を評価します。`pairwise_text_quality` は、モデルの応答を評価するために次の基準を使用します。

```
STEP 1: Analyze Response A based on all the Criteria provided, including Coherence, Fluency, Instruction following, Groundedness, and Verbosity. Provide assessment according to each criterion.
STEP 2: Analyze Response B based on all the Criteria provided, including Coherence, Fluency, Instruction following, Groundedness, and Verbosity. Provide assessment according to each criterion 
STEP 3: Compare the overall performance of Response A and Response B based on your analyses and assessment of each criterion 
STEP 4: Output your preference of "A", "SAME" or "B" to the pairwise_choice field according to the Rating Rubric.
STEP 5: Output your assessment reasoning in the explanation field, justifying your choice by highlighting the specific strengths and weaknesses of each response in terms of Text Quality
```

In [ ]:
# 「ペアワイズ テキスト品質」指標を作成
text_quality_prompt_template = MetricPromptTemplateExamples.get_prompt_template(
    "pairwise_text_quality"
)

pairwise_text_quality_metric = PairwiseMetric(
    metric="pairwise_text_quality",
    metric_prompt_template=text_quality_prompt_template,
    baseline_model=model_a,
)

データセットと評価基準を定義したので、評価ジョブを開始する準備が整いました。

注意: 下のセルは実行完了までに 10 〜 15 分かかります。

In [ ]:
pairwise_text_quality_eval_task = EvalTask(
    dataset=pairwise_eval_dataset,
    metrics=[pairwise_text_quality_metric],
    experiment=EXPERIMENT_NAME,
)

# ペアワイズ比較の候補モデルを指定
pairwise_text_quality_result = pairwise_text_quality_eval_task.evaluate(
    model=model_b,
    prompt_template=prompt_template,
)

上記の実験が実行されたら、`display_eval_report` ヘルパー関数を使用して結果を表示できます。

In [ ]:
display_eval_report(
    (
        "Side-by-side EvalTask",
        pairwise_text_quality_result.summary_metrics,
        pairwise_text_quality_result.metrics_table,
    )
)

ノートブックで実行した上記のすべての評価実験には、Agent Platform UI の [Experiments タブ](https://console.cloud.google.com/agent-platform/experiments/experiments) からアクセスできます。

Copyright 2024 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.